# R7 — Financial-Loss Reconstruction Ablation

Training-objective ablation only: universe, R1 primary target (`target_abs_1pct_next_week`), the authoritative full engineered feature set (34 features), the R1 cold-restart-every-fold walk-forward cadence, and the NN v1/NN v2 architectures/hyperparameters are all held fixed and reused verbatim (R6's stateful cadence is explicitly NOT used, per its R6_NEGATIVE result). Only the training loss changes, across exactly three predeclared candidates: `R7_BCE_CONTROL` (A, unchanged from R1), `R7_RETURN_WEIGHTED_BCE` (B), and `R7_BCE_PLUS_RETURN_UTILITY` (C). B and C are bounded, source-supported *reconstruction candidates* -- not claims of the paper's unpublished exact financial loss. See `docs/weekly_sector_rotation_r7_financial_loss_test.md` for the full write-up and `r7_financial_losses.py` for the shared, unit-tested loss/metric implementations this notebook imports. Does not implement R8.

In [1]:
"""
R7 full pipeline (scratchpad validation run). Once verified, transcribed into
r7_financial_loss_ablation.ipynb and executed there for the committed
artifacts. Loss-function ablation only: three predeclared training objectives
(A=BCE control, B=return-weighted BCE, C=BCE+return-utility hybrid) under the
exact R1 cold-restart walk-forward cadence, same features/target/architecture.
"""
import json
import sys
import time
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()
REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

sys.path.insert(0, str(REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation"))
import r7_financial_losses as rfl  # noqa: E402

SECTOR_ETFS = rfl.SECTOR_ETFS
FULL_FEATURE_COLS = rfl.FULL_FEATURE_COLS
RANK_K_VALUES = [2, 3]
BLOCK_SIZE = rfl.BLOCK_SIZE
N_BOOTSTRAP = rfl.N_BOOTSTRAP
BOOTSTRAP_SEED = rfl.BOOTSTRAP_SEED
OBJECTIVES = ["A", "B", "C"]

# ---------------------------------------------------------------------------

## 1. Load panel + R1 primary target, build the authoritative feature set (imported from `r7_financial_losses.py`), and reproduce/verify R1's exact fold schedule (with the same guard assertion R3/R4/R6 introduced).

In [2]:
# 1. Load panel + R1 target, build the authoritative feature set (imported).
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_open_to_close_return",
                "next_week_spy_return"]],
    on=["date", "symbol"], how="left",
)

assert set(FULL_FEATURE_COLS).issubset(set(FEATURE_COLUMNS))

md = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md["label_binary"] = md["target_abs_1pct_next_week"].astype(int)
md["year"] = md["date"].dt.year
years_all = sorted(md["year"].unique())
assert years_all == list(range(2005, 2027))
test_years = rfl.derive_r1_fold_schedule(years_all)
assert test_years == list(range(2008, 2027)), \
    f"fold schedule diverges from validated R1 2008-2026 schedule: {test_years}"
print(f"[{time.time()-t0:.0f}s] data loaded, {len(md)} rows, {len(FULL_FEATURE_COLS)} features, "
      f"test_years={test_years[0]}-{test_years[-1]} ({len(test_years)} folds)")

# ---------------------------------------------------------------------------

[0s] data loaded, 12253 rows, 34 features, test_years=2008-2026 (19 folds)


## 2. Model definitions -- verbatim R1 `SmallMLP` (NN v1) / `WideSingleLayerMLP` seed-ensemble (NN v2) -- plus the walk-forward evaluator: identical cold-restart-every-fold cadence to R1 (fresh model + fresh scaler every fold, no weight persistence, same fit/val/test split), with the only change being the training-loss objective. Checkpoint selection (early stopping) always uses plain BCE validation loss regardless of the training objective (Step 6's predeclared rule) -- the training objective and the model-selection metric are never both changed at once.

In [3]:
# 2. Model definitions -- verbatim R1 hidden config / hyperparameters.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


BCE_VAL_LOSS = nn.BCEWithLogitsLoss()


def train_fold_mlp(X_fit, y_fit, r_fit, X_val, y_val, n_features, objective, seed=NN_SEED, v2=False):
    """Identical R1 training loop (full-batch gradient descent, same epoch/patience
    budget); the only change is the training-loss objective (Step 6: checkpoint
    selection / early stopping ALWAYS uses plain BCE validation loss for every
    objective A/B/C, predeclared, so training-objective and model-selection-metric
    never both change at once)."""
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features) if v2 else SmallMLP(n_features)
    lr = V2_LEARNING_RATE if v2 else LEARNING_RATE
    wd = V2_WEIGHT_DECAY if v2 else WEIGHT_DECAY
    max_epochs = V2_MAX_EPOCHS if v2 else MAX_EPOCHS
    patience = V2_PATIENCE if v2 else PATIENCE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    r_fit_t = torch.tensor(r_fit, dtype=torch.float32) if r_fit is not None else None
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(max_epochs):
        model.train(); optimizer.zero_grad()
        loss = rfl.compute_training_loss(objective, model(X_fit_t), y_fit_t, r_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = BCE_VAL_LOSS(model(X_val_t), y_val_t).item()  # ALWAYS plain BCE for model selection
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def walk_forward_evaluate(md, years_, test_years_, feature_cols, objective, v2=False):
    oof_frames = []
    fold_rows = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        rfl.assert_train_before_test(train_years_available, test_year)  # Step 5 leakage guard

        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year

        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()
        r_fit = md.loc[fit_mask, "next_week_open_to_close_return"].to_numpy() if objective in ("B", "C") else None

        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_fold_mlp(X_fit, y_fit, r_fit, X_val, y_val, len(feature_cols), objective,
                                        seed=SEED_BASE + seed_offset, v2=True)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_fold_mlp(X_fit, y_fit, r_fit, X_val, y_val, len(feature_cols), objective,
                                    seed=NN_SEED, v2=False)
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = md.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                   "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)

        y_test = md.loc[test_mask, "label_binary"]
        fold_auc = roc_auc_score(y_test, predicted_proba) if y_test.nunique() > 1 else np.nan
        fold_prauc = average_precision_score(y_test, predicted_proba) if y_test.nunique() > 1 else np.nan
        fold_brier = brier_score_loss(y_test, predicted_proba)
        fold_rows.append({
            "test_year": test_year, "n_train": int(train_mask.sum()), "n_fit": int(fit_mask.sum()),
            "n_val": int(val_mask.sum()), "n_test": int(test_mask.sum()),
            "train_positive_rate": float(y_fit.mean()), "test_positive_rate": float(y_test.mean()),
            "fold_auc": fold_auc, "fold_pr_auc": fold_prauc, "fold_brier": fold_brier,
        })
    oof = pd.concat(oof_frames, ignore_index=True)
    return oof, pd.DataFrame(fold_rows)


print(f"[{time.time()-t0:.0f}s] Step 2 (model defs + walk-forward evaluator) done.")

# ---------------------------------------------------------------------------

[0s] Step 2 (model defs + walk-forward evaluator) done.


## 3. Run all 3 objectives (A=BCE control, B=return-weighted BCE, C=BCE+return-utility) x 2 models x 19 folds, then verify Objective A exactly reproduces R1's saved full-feature-set AUCs (float32 tolerance) -- confirming the walk-forward/data/seed pipeline is unchanged from R1.

In [4]:
# 3. Run all 3 objectives x 2 models, all 19 folds.
# ---------------------------------------------------------------------------
oof_store = {}
fold_rows_all = []
for model_name, v2 in [("NN_v1", False), ("NN_v2", True)]:
    for objective in OBJECTIVES:
        oof, folds = walk_forward_evaluate(md, years_all, test_years, FULL_FEATURE_COLS, objective, v2=v2)
        oof_store[(model_name, objective)] = oof
        folds.insert(0, "model", model_name); folds.insert(1, "objective", objective)
        fold_rows_all.append(folds)
        auc = roc_auc_score(oof.label_binary, oof.predicted_proba)
        print(f"[{time.time()-t0:.0f}s] {model_name} objective={objective} "
              f"({rfl.OBJECTIVE_LABELS[objective]}): pooled_auc={auc:.4f}")

r7_fold_metrics = pd.concat(fold_rows_all, ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 3 (all runs) done.")

# ---------------------------------------------------------------------------
# 3b. Sanity check: Objective A (BCE control) must reproduce R1's saved
#     full-feature-set AUCs exactly (same code path, same seeds, same data).
# ---------------------------------------------------------------------------
R1_SAVED_AUC = {"NN_v1": 0.5680100994943317, "NN_v2": 0.5735730758971164}
for model_name, expected in R1_SAVED_AUC.items():
    oof_a = oof_store[(model_name, "A")]
    actual = roc_auc_score(oof_a.label_binary, oof_a.predicted_proba)
    diff = abs(actual - expected)
    print(f"  sanity check {model_name} Objective A vs R1 saved: actual={actual:.6f} expected={expected:.6f} diff={diff:.6f}")
    assert diff < 1e-4, f"Objective A (BCE control) does not reproduce R1's saved AUC for {model_name}"
print(f"[{time.time()-t0:.0f}s] Step 3b (BCE control reproduces R1 exactly) verified.")

# ---------------------------------------------------------------------------

[4s] NN_v1 objective=A (R7_BCE_CONTROL): pooled_auc=0.5680


[7s] NN_v1 objective=B (R7_RETURN_WEIGHTED_BCE): pooled_auc=0.5672


[11s] NN_v1 objective=C (R7_BCE_PLUS_RETURN_UTILITY): pooled_auc=0.5689


[26s] NN_v2 objective=A (R7_BCE_CONTROL): pooled_auc=0.5736


[40s] NN_v2 objective=B (R7_RETURN_WEIGHTED_BCE): pooled_auc=0.5744


[55s] NN_v2 objective=C (R7_BCE_PLUS_RETURN_UTILITY): pooled_auc=0.5741
[55s] Step 3 (all runs) done.
  sanity check NN_v1 Objective A vs R1 saved: actual=0.568013 expected=0.568010 diff=0.000003
  sanity check NN_v2 Objective A vs R1 saved: actual=0.573571 expected=0.573573 diff=0.000002
[55s] Step 3b (BCE control reproduces R1 exactly) verified.


## 4. Aggregate metrics, Top-K, calibration (Brier / mean predicted probability / observed rate / decile reliability buckets), and score-distribution diagnostics -- reusing the shared `topk_metrics_and_series` utility from `r7_financial_losses.py`.

In [5]:
# 4. Aggregate metrics, Top-K, calibration, score-distribution -- reusing the
#    shared topk_metrics_and_series / block_bootstrap_ci from r7_financial_losses.
# ---------------------------------------------------------------------------
topk_metrics_and_series = rfl.topk_metrics_and_series
block_bootstrap_ci = rfl.block_bootstrap_ci


def _weekly_yscore_lists(oof: pd.DataFrame):
    dates = sorted(oof["date"].unique())
    y_list, s_list = [], []
    for d in dates:
        g = oof[oof["date"] == d]
        y_list.append(g["label_binary"].to_numpy()); s_list.append(g["predicted_proba"].to_numpy())
    return dates, y_list, s_list


def block_bootstrap_auc_edge(oof: pd.DataFrame, block_size: int = BLOCK_SIZE, n_boot: int = N_BOOTSTRAP,
                              seed: int = BOOTSTRAP_SEED) -> dict:
    dates, y_list, s_list = _weekly_yscore_lists(oof)
    n = len(dates); rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size)); max_start = n - block_size
    point_auc = roc_auc_score(oof["label_binary"], oof["predicted_proba"])
    boot_aucs = np.full(n_boot, np.nan)
    for b in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y = np.concatenate([y_list[i] for i in idxs]); s = np.concatenate([s_list[i] for i in idxs])
        if len(np.unique(y)) > 1:
            boot_aucs[b] = roc_auc_score(y, s)
    valid = boot_aucs[~np.isnan(boot_aucs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    return {"point_estimate": point_auc - 0.5, "ci_lower": lower - 0.5, "ci_upper": upper - 0.5,
            "n_weeks": n, "significant": bool((lower - 0.5) > 0 or (upper - 0.5) < 0)}


def paired_block_bootstrap_auc_diff(oof_a: pd.DataFrame, oof_b: pd.DataFrame, block_size: int = BLOCK_SIZE,
                                     n_boot: int = N_BOOTSTRAP, seed: int = BOOTSTRAP_SEED) -> dict:
    common_dates = sorted(set(oof_a["date"].unique()) & set(oof_b["date"].unique()))
    oof_a = oof_a[oof_a["date"].isin(common_dates)]; oof_b = oof_b[oof_b["date"].isin(common_dates)]
    dates_a, y_list_a, s_list_a = _weekly_yscore_lists(oof_a)
    dates_b, y_list_b, s_list_b = _weekly_yscore_lists(oof_b)
    assert dates_a == dates_b
    n = len(dates_a); rng = np.random.default_rng(seed)
    n_blocks_needed = int(ceil(n / block_size)); max_start = n - block_size
    point_auc_a = roc_auc_score(oof_a["label_binary"], oof_a["predicted_proba"])
    point_auc_b = roc_auc_score(oof_b["label_binary"], oof_b["predicted_proba"])
    boot_diffs = np.full(n_boot, np.nan)
    for it in range(n_boot):
        starts = rng.integers(0, max_start + 1, size=n_blocks_needed)
        idxs = np.concatenate([np.arange(s, s + block_size) for s in starts])[:n]
        y_a = np.concatenate([y_list_a[i] for i in idxs]); s_a = np.concatenate([s_list_a[i] for i in idxs])
        y_b = np.concatenate([y_list_b[i] for i in idxs]); s_b = np.concatenate([s_list_b[i] for i in idxs])
        if len(np.unique(y_a)) > 1 and len(np.unique(y_b)) > 1:
            boot_diffs[it] = roc_auc_score(y_a, s_a) - roc_auc_score(y_b, s_b)
    valid = boot_diffs[~np.isnan(boot_diffs)]
    lower, upper = np.percentile(valid, [2.5, 97.5])
    point = point_auc_a - point_auc_b
    return {"point_estimate": point, "ci_lower": lower, "ci_upper": upper, "n_weeks": n,
            "significant": bool(lower > 0 or upper < 0)}


def paired_series_diff_bootstrap(oof_a, oof_b, base_rate_a, base_rate_b, k):
    _, series_a = topk_metrics_and_series(oof_a, base_rate_a, k)
    _, series_b = topk_metrics_and_series(oof_b, base_rate_b, k)
    dfa = pd.DataFrame({"date": series_a["date"], "precision": series_a["precision"],
                         "sel_minus_all": series_a["selected_ret"] - series_a["all_sector_mean"],
                         "spread": series_a["selected_ret"] - series_a["bottomk_mean"],
                         "spy_edge": series_a["selected_ret"] - series_a["spy_ret"]}).set_index("date")
    dfb = pd.DataFrame({"date": series_b["date"], "precision": series_b["precision"],
                         "sel_minus_all": series_b["selected_ret"] - series_b["all_sector_mean"],
                         "spread": series_b["selected_ret"] - series_b["bottomk_mean"],
                         "spy_edge": series_b["selected_ret"] - series_b["spy_ret"]}).set_index("date")
    common = dfa.join(dfb, lsuffix="_a", rsuffix="_b", how="inner")
    out = {}
    for qname, col in [("precision_minus_base_diff", "precision"), ("topk_minus_allsector_diff", "sel_minus_all"),
                        ("topk_minus_bottomk_diff", "spread"), ("topk_minus_spy_diff", "spy_edge")]:
        diff_series = common[f"{col}_a"] - common[f"{col}_b"]
        out[qname] = block_bootstrap_ci(diff_series)
    return out


aggregate_rows = []
topk_rows = []
calibration_rows = []
score_dist_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for objective in OBJECTIVES:
        oof = oof_store[(model_name, objective)]
        folds = r7_fold_metrics[(r7_fold_metrics.model == model_name) & (r7_fold_metrics.objective == objective)]
        auc = roc_auc_score(oof.label_binary, oof.predicted_proba)
        prauc = average_precision_score(oof.label_binary, oof.predicted_proba)
        brier = brier_score_loss(oof.label_binary, oof.predicted_proba)
        base_rate = oof.label_binary.mean()
        aggregate_rows.append({
            "model": model_name, "objective": objective, "objective_label": rfl.OBJECTIVE_LABELS[objective],
            "n_oof_rows": len(oof), "pooled_auc": auc, "pooled_pr_auc": prauc, "pooled_brier": brier,
            "mean_fold_auc": folds.fold_auc.mean(), "median_fold_auc": folds.fold_auc.median(),
            "std_fold_auc": folds.fold_auc.std(), "best_fold_auc": folds.fold_auc.max(),
            "worst_fold_auc": folds.fold_auc.min(), "frac_folds_above_0.5": float((folds.fold_auc > 0.5).mean()),
            "base_positive_rate": base_rate,
        })

        for k in RANK_K_VALUES:
            row, _ = topk_metrics_and_series(oof, base_rate, k)
            row.update({"model": model_name, "objective": objective})
            topk_rows.append(row)

        # calibration: scalar summary + 10 reliability buckets
        calibration_rows.append({
            "model": model_name, "objective": objective, "row_type": "summary",
            "bucket_id": np.nan, "bucket_mean_pred": np.nan, "bucket_observed_rate": np.nan, "bucket_n": np.nan,
            "brier": brier, "mean_predicted_proba": float(oof.predicted_proba.mean()),
            "observed_positive_rate": float(base_rate),
        })
        bucket_ids = pd.qcut(oof.predicted_proba, 10, labels=False, duplicates="drop")
        for b in sorted(pd.unique(bucket_ids.dropna())):
            mask = bucket_ids == b
            calibration_rows.append({
                "model": model_name, "objective": objective, "row_type": "calibration_bucket",
                "bucket_id": int(b), "bucket_mean_pred": float(oof.loc[mask, "predicted_proba"].mean()),
                "bucket_observed_rate": float(oof.loc[mask, "label_binary"].mean()),
                "bucket_n": int(mask.sum()), "brier": np.nan, "mean_predicted_proba": np.nan,
                "observed_positive_rate": np.nan,
            })

        # score-distribution diagnostics
        weekly_top_scores = {1: [], 2: [], 3: []}
        for date, g in oof.groupby("date"):
            if len(g) < 11:
                continue
            sorted_scores = g["predicted_proba"].sort_values(ascending=False).to_numpy()
            for kk in (1, 2, 3):
                weekly_top_scores[kk].append(sorted_scores[:kk].mean())
        pos_scores = oof.loc[oof.label_binary == 1, "predicted_proba"]
        neg_scores = oof.loc[oof.label_binary == 0, "predicted_proba"]
        score_dist_rows.append({
            "model": model_name, "objective": objective,
            "mean_score": float(oof.predicted_proba.mean()), "std_score": float(oof.predicted_proba.std()),
            "p5_score": float(oof.predicted_proba.quantile(0.05)), "median_score": float(oof.predicted_proba.median()),
            "p95_score": float(oof.predicted_proba.quantile(0.95)),
            "avg_weekly_top1_score": float(np.mean(weekly_top_scores[1])),
            "avg_weekly_top2_score": float(np.mean(weekly_top_scores[2])),
            "avg_weekly_top3_score": float(np.mean(weekly_top_scores[3])),
            "mean_score_positive_class": float(pos_scores.mean()), "mean_score_negative_class": float(neg_scores.mean()),
            "positive_minus_negative_separation": float(pos_scores.mean() - neg_scores.mean()),
        })

r7_aggregate_metrics = pd.DataFrame(aggregate_rows)
r7_topk_all = pd.DataFrame(topk_rows)
r7_calibration_metrics = pd.DataFrame(calibration_rows)
r7_score_distribution = pd.DataFrame(score_dist_rows)
print(f"[{time.time()-t0:.0f}s] Step 4 (aggregate/topk/calibration/score-distribution) done.")
print(r7_aggregate_metrics[["model", "objective", "pooled_auc", "mean_fold_auc", "std_fold_auc"]].to_string())

# ---------------------------------------------------------------------------

[60s] Step 4 (aggregate/topk/calibration/score-distribution) done.
   model objective  pooled_auc  mean_fold_auc  std_fold_auc
0  NN_v1         A    0.568013       0.578866      0.056395
1  NN_v1         B    0.567204       0.571087      0.061477
2  NN_v1         C    0.568873       0.579492      0.056394
3  NN_v2         A    0.573571       0.584769      0.054392
4  NN_v2         B    0.574409       0.583336      0.057554
5  NN_v2         C    0.574139       0.584950      0.055503


## 5. Standalone block-bootstrap CIs (each objective vs. chance).

In [6]:
# 5. Standalone bootstrap (each objective vs chance).
# ---------------------------------------------------------------------------
bootstrap_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for objective in OBJECTIVES:
        oof = oof_store[(model_name, objective)]
        base_rate = oof.label_binary.mean()
        b = block_bootstrap_auc_edge(oof)
        bootstrap_rows.append({"model": model_name, "objective": objective, "k": np.nan,
                                "quantity": "auc_edge_over_0.5", **b})
        for k in RANK_K_VALUES:
            _, series = topk_metrics_and_series(oof, base_rate, k)
            b = block_bootstrap_ci(series["precision"])
            bootstrap_rows.append({"model": model_name, "objective": objective, "k": k,
                                    "quantity": "precision_minus_base",
                                    "point_estimate": b["point_estimate"] - base_rate,
                                    "ci_lower": b["ci_lower"] - base_rate, "ci_upper": b["ci_upper"] - base_rate,
                                    "n_weeks": b["n_weeks"], "significant": b["significant"]})
            for qname, diff in [
                ("selected_minus_all_sector_mean", series["selected_ret"] - series["all_sector_mean"]),
                ("selected_minus_spy", series["selected_ret"] - series["spy_ret"]),
                ("topk_minus_bottomk_spread", series["selected_ret"] - series["bottomk_mean"]),
            ]:
                b = block_bootstrap_ci(diff)
                bootstrap_rows.append({"model": model_name, "objective": objective, "k": k, "quantity": qname, **b})
r7_bootstrap_ci = pd.DataFrame(bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 5 (standalone bootstrap) done.")

# ---------------------------------------------------------------------------

[139s] Step 5 (standalone bootstrap) done.


## 6. Paired vs.-BCE-control block bootstrap -- the primary R7 statistical test, on identical out-of-fold weeks for each model.

In [7]:
# 6. Paired vs BCE control bootstrap -- the primary R7 statistical test.
# ---------------------------------------------------------------------------
paired_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    oof_bce = oof_store[(model_name, "A")]
    base_bce = oof_bce.label_binary.mean()
    for objective in ["B", "C"]:
        oof_fin = oof_store[(model_name, objective)]
        base_fin = oof_fin.label_binary.mean()
        auc_diff = paired_block_bootstrap_auc_diff(oof_fin, oof_bce)
        paired_rows.append({"model": model_name, "objective": objective, "k": np.nan, "quantity": "auc", **auc_diff})
        for k in RANK_K_VALUES:
            diffs = paired_series_diff_bootstrap(oof_fin, oof_bce, base_fin, base_bce, k)
            for qname, b in diffs.items():
                paired_rows.append({"model": model_name, "objective": objective, "k": k, "quantity": qname, **b})
        print(f"[{time.time()-t0:.0f}s] {model_name} objective={objective} vs BCE: "
              f"auc_diff={auc_diff['point_estimate']:.4f} (sig={auc_diff['significant']})")

r7_vs_bce_paired = pd.DataFrame(paired_rows)
print(f"[{time.time()-t0:.0f}s] Step 6 (paired vs BCE bootstrap) done.")

# ---------------------------------------------------------------------------

[164s] NN_v1 objective=B vs BCE: auc_diff=-0.0008 (sig=False)


[189s] NN_v1 objective=C vs BCE: auc_diff=0.0009 (sig=False)


[213s] NN_v2 objective=B vs BCE: auc_diff=0.0008 (sig=False)


[238s] NN_v2 objective=C vs BCE: auc_diff=0.0006 (sig=False)
[238s] Step 6 (paired vs BCE bootstrap) done.


## 7. Sample audit (fold-level detail, confirming the Step 5 leakage guard passed for every fold).

In [8]:
# 7. Sample audit (fold-level detail + leakage-guard confirmation).
# ---------------------------------------------------------------------------
r7_sample_audit = r7_fold_metrics.copy()
r7_sample_audit["leakage_guard"] = "assert_train_before_test passed for every fold (Step 5)"

# ---------------------------------------------------------------------------

## 8. Bengio direct-objective feasibility analysis (Step 16) -- documented reasoning only; no implementation attempt, per the mandate.

In [9]:
# 8. Bengio direct-objective feasibility analysis (Step 16) -- documented
#    reasoning only, no implementation attempt.
# ---------------------------------------------------------------------------
bengio_feasibility = {
    "classification": "DIRECT_BENGIO_OBJECTIVE_DEFERRED_SOURCE_AMBIGUITY",
    "recovered_criterion": (
        "C = sum_t [ log(sum_i r[t,i]*w[t,i]) + log(1 - sum_i c_i*|w[t,i]-w'[(t-1),i]|) ], "
        "w'[(t-1),i] = w[(t-1),i]*r[(t-1),i] / sum_j w[(t-1),j]*r[(t-1),j]"
    ),
    "unresolved_assumptions_required_simultaneously": [
        "Differentiable mapping from 11 independent scalar model scores to a valid portfolio weight "
        "vector w[t,:] is unspecified -- softmax over scores, rank-based allocation, and top-K equal-"
        "weight are all plausible but materially different choices the source does not disambiguate.",
        "Whether cash/unallocated capital is permitted (sum_i w[t,i] <= 1) or full investment is forced "
        "(sum_i w[t,i] == 1) is not stated.",
        "Long-only vs. signed (short-permitting) weights is not stated; our system has never modeled "
        "shorting anywhere in R1-R6.",
        "The per-asset transaction cost coefficients c_i are not published for this universe/paper.",
        "How the drifted prior weight w'[(t-1),i] is initialized for the very first batch/fold (no "
        "t-1 portfolio exists yet) is unspecified.",
        "Portfolio state (w[(t-1),:]) must persist ACROSS weekly batches for the turnover term to be "
        "computable -- this requires temporally-ordered, non-shuffled training, which is a structural "
        "change to the current per-row i.i.d. training loop (Step 5/8 of R6 already showed stateful "
        "sequencing changes are themselves a confounding variable, not something to introduce silently "
        "inside a loss-function ablation).",
        "R7's architecture produces one independent scalar score per (week, sector) row -- there is no "
        "single forward pass that jointly emits all 11 weights for a week (that would be R4's MIMO "
        "geometry, already tested and found R4_NULL) -- so the portfolio-return log-term literally "
        "cannot be computed from the current per-row training batches without first resolving the MIMO "
        "question again.",
    ],
    "reasoning": (
        "Implementing the recovered Bengio criterion faithfully would require resolving at least six "
        "materially independent unknowns simultaneously (weight mapping, cash/leverage, long-only vs "
        "signed, transaction costs, prior-weight initialization, and sequential/stateful batching), and "
        "would also implicitly require re-adopting either R4's MIMO output geometry (R4_NULL) or R6's "
        "stateful sequential training (R6_NEGATIVE) -- both already tested and rejected on their own "
        "terms in this project. Bundling all of these into a single R7 loss-ablation would make any "
        "result impossible to attribute to the loss function specifically, violating R7's bounded-"
        "reconstruction mandate. Objectives B and C are the defensible, single-assumption-at-a-time "
        "reconstruction candidates that remain answerable within the current architecture; the full "
        "Bengio portfolio objective is deferred, not attempted, in this pass."
    ),
}
with open(OUTPUTS_ROOT / "r7_bengio_feasibility.json", "w") as f:
    json.dump(bengio_feasibility, f, indent=2)
print(f"[{time.time()-t0:.0f}s] Step 8 (Bengio feasibility analysis) done: "
      f"{bengio_feasibility['classification']}")

# ---------------------------------------------------------------------------

[238s] Step 8 (Bengio feasibility analysis) done: DIRECT_BENGIO_OBJECTIVE_DEFERRED_SOURCE_AMBIGUITY


## 9. Save all required R7 artifacts.

In [10]:
# 9. Save all artifacts.
# ---------------------------------------------------------------------------
r7_fold_metrics.to_csv(OUTPUTS_ROOT / "r7_fold_metrics.csv", index=False)
r7_aggregate_metrics.to_csv(OUTPUTS_ROOT / "r7_aggregate_metrics.csv", index=False)
r7_topk_all[r7_topk_all["k"] == 2].to_csv(OUTPUTS_ROOT / "r7_top2_metrics.csv", index=False)
r7_topk_all[r7_topk_all["k"] == 3].to_csv(OUTPUTS_ROOT / "r7_top3_metrics.csv", index=False)
r7_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r7_bootstrap_ci.csv", index=False)
r7_vs_bce_paired.to_csv(OUTPUTS_ROOT / "r7_vs_bce_paired_comparison.csv", index=False)
r7_calibration_metrics.to_csv(OUTPUTS_ROOT / "r7_calibration_metrics.csv", index=False)
r7_score_distribution.to_csv(OUTPUTS_ROOT / "r7_score_distribution.csv", index=False)
r7_sample_audit.to_csv(OUTPUTS_ROOT / "r7_sample_audit.csv", index=False)

summary = {
    "primary_feature_set": "A+B+C+D+E+F+G+H+I (full engineered set, 34 features)",
    "test_years": [int(test_years[0]), int(test_years[-1])],
    "objectives": {k: v for k, v in rfl.OBJECTIVE_LABELS.items()},
    "fixed_hyperparameters": {"W_CAP": rfl.W_CAP, "R_CAP": rfl.R_CAP, "LAMBDA": rfl.LAMBDA},
    "pooled_auc": {
        f"{m}_{o}": float(roc_auc_score(oof_store[(m, o)].label_binary, oof_store[(m, o)].predicted_proba))
        for m in ["NN_v1", "NN_v2"] for o in OBJECTIVES
    },
    "bengio_feasibility_classification": bengio_feasibility["classification"],
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r7_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2))

[238s] ALL DONE.
{
  "primary_feature_set": "A+B+C+D+E+F+G+H+I (full engineered set, 34 features)",
  "test_years": [
    2008,
    2026
  ],
  "objectives": {
    "A": "R7_BCE_CONTROL",
    "B": "R7_RETURN_WEIGHTED_BCE",
    "C": "R7_BCE_PLUS_RETURN_UTILITY"
  },
  "fixed_hyperparameters": {
    "W_CAP": 3.0,
    "R_CAP": 3.0,
    "LAMBDA": 0.1
  },
  "pooled_auc": {
    "NN_v1_A": 0.5680133558294063,
    "NN_v1_B": 0.5672039927831907,
    "NN_v1_C": 0.5688734907272295,
    "NN_v2_A": 0.5735714959002163,
    "NN_v2_B": 0.5744093759636536,
    "NN_v2_C": 0.5741394855175557
  },
  "bengio_feasibility_classification": "DIRECT_BENGIO_OBJECTIVE_DEFERRED_SOURCE_AMBIGUITY",
  "runtime_seconds": 238.1551742553711
}
